# 1. Title and context — Utilities validation (archival)

**Question:** Do the **helper functions** inside the public engine (`evaluate_gates`, compensatory score, abstention rate, batch hash) behave consistently on a **fixed feature vector**?

**Pipeline:** step 2 in `notebook_plan.json`; produces `outputs/tables/utilities_validation.csv` only.


# 2. Why this matters

Reviewers often inspect **intermediate** quantities—not only final APPROVE/REJECT. This notebook surfaces gate pass bits, compensatory score, SCM abstention mapping at two calibration points, and a one-case batch digest—all from **`corrected_public_engine_v1_1.py`** without duplicating formulas.


# 3. Deterministic setup

Same assumptions as notebook 01: declared deps including **ipywidgets**, hash seed from harness config, **restart-and-run-all** safe. **Disk outputs** use only `ARCHIVAL_FEATURES`; the dropdown explores alternate **predefined** feature dicts in-memory only.


In [1]:
from __future__ import annotations

import csv
import os
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display


def repo_root() -> Path:
    """Resolve repository root from env override or parent-anchor scan."""
    env_root = os.environ.get("EAA_REPO_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "engine" / "corrected_public_engine_v1_1.py").is_file():
            return p

    start = Path.cwd().resolve()
    anchors = [
        Path("engine") / "corrected_public_engine_v1_1.py",
        Path("config") / "notebook_plan.json",
        Path("reproduce_all.py"),
    ]
    for p in (start, *start.parents):
        if all((p / a).exists() for a in anchors):
            return p
    raise RuntimeError(
        f"Repository root not found from cwd={start}. Expected anchors: {anchors}"
    )


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "engine"))
import corrected_public_engine_v1_1 as eng


# 4. Inputs — archival vector + exploration presets

`ARCHIVAL_FEATURES` feeds the CSV contract. `FEATURE_PRESETS` is a **closed** set for the widget.


In [2]:
ARCHIVAL_FEATURES = {
    "intrinsic_safety": 0.60,
    "evidence_strength": 0.58,
    "bias_harm_index": 0.42,
    "uncertainty_calibration": 0.55,
    "traceability_integrity": 0.56,
}

FEATURE_PRESETS = {
    "Archival contract vector": ARCHIVAL_FEATURES,
    "Higher safety emphasis": {
        "intrinsic_safety": 0.75,
        "evidence_strength": 0.58,
        "bias_harm_index": 0.42,
        "uncertainty_calibration": 0.55,
        "traceability_integrity": 0.56,
    },
}


# 5. Modular engine calls


In [3]:
def summarize_utilities(features: dict) -> dict:
    profile = eng.CANONICAL_THRESHOLD_PROFILES["moderate"]
    gates = eng.evaluate_gates(features, profile)
    comp = eng.compute_compensatory_score(features)
    am = eng.compute_abstention_rate(0.5)
    auc = eng.compute_abstention_rate(features["uncertainty_calibration"])
    return {
        "all_gates_pass": gates["all_gates_pass"],
        "binding_n": len(gates["binding_constraints"]),
        "compensatory_score": round(comp, 6),
        "abstention_mid": round(am, 6),
        "abstention_uc": round(auc, 6),
    }


# 6. Interactive exploration (bounded, non-destructive)

Select a predefined scenario below. This changes on-screen results only; canonical outputs remain fixed.


In [4]:
u_out = widgets.Output(layout={"border": "1px solid #ccc", "padding": "6px"})
udd = widgets.Dropdown(
    options=list(FEATURE_PRESETS.keys()),
    value="Archival contract vector",
    description="Feature preset:",
    style={"description_width": "initial"},
)


def _render_u(label: str) -> None:
    with u_out:
        clear_output(wait=True)
        feats = FEATURE_PRESETS[label]
        s = summarize_utilities(feats)
        print(f"Preset: {label}")
        for k, v in s.items():
            print(f"  {k}: {v}")


def _on_u(change) -> None:
    if change.get("name") == "value" and change.get("new") is not None:
        _render_u(change["new"])


udd.observe(_on_u, names="value")
_render_u(udd.value)
display(udd)
display(u_out)


Dropdown(description='Feature preset:', options=('Archival contract vector', 'Higher safety emphasis'), style=…

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

# 7. Output generation — `utilities_validation.csv` only


In [5]:
FEATURES = ARCHIVAL_FEATURES

profile = eng.CANONICAL_THRESHOLD_PROFILES["moderate"]
gates = eng.evaluate_gates(FEATURES, profile)
comp_score = eng.compute_compensatory_score(FEATURES)
abstention_mid = eng.compute_abstention_rate(0.5)
abstention_uc = eng.compute_abstention_rate(FEATURES["uncertainty_calibration"])

fixture = {"case_id": "hash_fixture", "features": FEATURES}
batch = eng.evaluate_batch([fixture], profile_names=["moderate"], mode=eng.MODE_REPLAY)
batch_hash = eng.hash_output(batch)

rows = [
    ("evaluate_gates_all_pass", gates["all_gates_pass"]),
    ("binding_constraints_count", len(gates["binding_constraints"])),
    ("compensatory_score", round(comp_score, 6)),
    ("abstention_rate_at_uc_0.5", round(abstention_mid, 6)),
    ("abstention_rate_case_uc", round(abstention_uc, 6)),
    ("hash_batch_fixture_sha256", batch_hash),
]

out_path = ROOT / "outputs" / "tables" / "utilities_validation.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["check_name", "value"])
    for name, val in rows:
        w.writerow([name, val])

print("Utilities validation CSV written.")


Utilities validation CSV written.


# 8. Interpretation

Each CSV row names a function in the engine module. The batch hash row ties **`evaluate_batch`** to **`hash_output`** for a minimal fixture—useful when comparing interactive exploration to the locked file.

# 9. Reproducibility

Contract rows depend only on `ARCHIVAL_FEATURES`. Widgets are optional for understanding; **`reproduce_all.py`** runs without manual clicks. Deterministic shared-core validation notebook.
